# Restricted-input method reference
Source: analysis/inter-park/road_connectivity.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas
import geopandas as gpd
# connectivity_from_geojson.py
# Requirements: geopandas, shapely>=2.0, networkx, numpy, pandas, scikit-learn, scipy
# pip install geopandas shapely networkx numpy pandas scikit-learn scipy

from __future__ import annotations
import math
from dataclasses import dataclass
from typing import Dict, Any, Literal, Optional

import numpy as np
import networkx as nx
from networkx.algorithms.efficiency_measures import global_efficiency
from shapely.geometry import LineString, MultiLineString, Point
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
@dataclass
class ConnectivityResult:
    one_value: float
    one_value_name: str
    metrics: Dict[str, Any]

def _explode_lines(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.explode(index_parts=False, ignore_index=True)
    # Keep only LineString
    gdf = gdf[gdf.geometry.type == "LineString"].copy()
    gdf.reset_index(drop=True, inplace=True)
    return gdf

def _segment_endpoints(lines: gpd.GeoDataFrame) -> np.ndarray:
    """Return Nx2 array of all segment endpoints (x, y)."""
    pts = []
    for geom in lines.geometry.values:
        if not isinstance(geom, LineString):
            continue
        x0, y0 = geom.coords[0]
        x1, y1 = geom.coords[-1]
        pts.append((x0, y0))
        pts.append((x1, y1))
    return np.asarray(pts, dtype=float)

def _cluster_endpoints(coords_xy: np.ndarray, snap_tolerance: float) -> np.ndarray:
    """
    Cluster endpoints so intersections close to each other become one node.
    Returns an array of cluster labels for each point in coords_xy.
    """
    if len(coords_xy) == 0:
        return np.array([], dtype=int)
    # eps in same units as CRS (metres after projection)
    labels = DBSCAN(eps=snap_tolerance, min_samples=1).fit(coords_xy).labels_
    return labels

def _build_graph(lines: gpd.GeoDataFrame, labels: np.ndarray, coords_xy: np.ndarray) -> nx.Graph:
    """
    Build an undirected, weighted graph.
    Each endpoint gets a cluster; each line connects its two endpoint clusters.
    Node position is the centroid of points in the cluster.
    Edge weight = segment length (metres).
    """
    G = nx.Graph()

    # Map: cluster_id -> list of points
    clusters: Dict[int, list[tuple[float, float]]] = {}
    for (x, y), lab in zip(coords_xy, labels):
        clusters.setdefault(lab, []).append((x, y))

    # Create nodes at cluster centroids
    for lab, points in clusters.items():
        arr = np.asarray(points)
        cx, cy = arr.mean(axis=0)
        G.add_node(int(lab), x=float(cx), y=float(cy))

    # Add edges
    # We need to iterate lines again and link their first and last endpoint clusters
    # Recompute per-line start/end cluster by matching to coords_xy order
    idx = 0
    for geom in lines.geometry.values:
        x0, y0 = geom.coords[0]
        x1, y1 = geom.coords[-1]

        lab0 = labels[idx]; lab1 = labels[idx + 1]
        idx += 2

        if lab0 == lab1:
            # Degenerate: both endpoints snapped to same cluster; skip
            continue

        length_m = float(geom.length)
        u, v = int(lab0), int(lab1)
        # If multiple segments connect same nodes, keep the shortest
        if G.has_edge(u, v):
            if length_m < G[u][v]["weight"]:
                G[u][v]["weight"] = length_m
        else:
            G.add_edge(u, v, weight=length_m)

    return G

def _algebraic_connectivity_safe(G: nx.Graph) -> Optional[float]:
    """Second smallest eigenvalue of Laplacian on the largest connected component."""
    if G.number_of_nodes() < 2:
        return None
    if not nx.is_connected(G):
        # work with giant component
        comp_nodes = max(nx.connected_components(G), key=len)
        H = G.subgraph(comp_nodes).copy()
    else:
        H = G
    try:
        # Requires scipy
        return float(nx.algebraic_connectivity(H, method="tracemin_lu"))
    except Exception:
        try:
            return float(nx.algebraic_connectivity(H))
        except Exception:
            return None

def _compute_metrics(G: nx.Graph) -> Dict[str, Any]:
    n = G.number_of_nodes()
    m = G.number_of_edges()
    comps = list(nx.connected_components(G))
    c = len(comps)
    n_lcc = max((len(s) for s in comps), default=0)
    lcc_ratio = (n_lcc / n) if n > 0 else 0.0
    density = nx.density(G) if n > 1 else 0.0
    avg_deg = (2 * m / n) if n > 0 else 0.0
    try:
        avg_clust = nx.average_clustering(G) if n > 1 else 0.0
    except Exception:
        avg_clust = None
    try:
        geff = global_efficiency(G) if n > 1 else 0.0
    except Exception:
        geff = None

    # Algebraic connectivity (on giant component)
    alg_conn = _algebraic_connectivity_safe(G)

    # Classical transport indices (planar assumption; use with care)
    # Cyclomatic number counts independent cycles (for c components)
    mu = m - n + c
    beta = (m / n) if n > 0 else 0.0
    gamma = (m / (3 * (n - 2))) if n > 2 else None
    alpha = ((m - n + c) / (2 * n - 5)) if n > 2 else None

    total_length_m = float(sum(d["weight"] for _, _, d in G.edges(data=True)))

    return {
        "nodes": n,
        "edges": m,
        "components": c,
        "largest_component_nodes": n_lcc,
        "giant_component_ratio": lcc_ratio,
        "edge_density": density,
        "average_degree": avg_deg,
        "average_clustering": avg_clust,
        "global_efficiency": geff,
        "algebraic_connectivity": alg_conn,
        "cyclomatic_number": mu,
        "beta_index": beta,
        "gamma_index": gamma,  # ~0..1 for planar graphs
        "alpha_index": alpha,  # ~0..1 for planar graphs
        "total_edge_length_m": total_length_m,
    }

def connectivity_from_geojson(
    gdf: gpd.GeoDataFrame,
    snap_tolerance_m: float = 5.0,
    one_value: Literal[
        "giant_component_ratio",
        "algebraic_connectivity",
        "edge_density",
        "average_clustering",
        "beta_index",
        "gamma_index",
        "alpha_index",
        "global_efficiency",
    ] = "giant_component_ratio",
) -> ConnectivityResult:
    """
    Compute a one-number connectivity score and a suite of metrics.
    - snap_tolerance_m: merges nearby endpoints into a single node (metres).
    - one_value: choose which single metric to report as the 'connectivity' value.
    """
    lines = _explode_lines(gdf)

    # Pre-simplify tiny dangling fragments (optional, conservative)
    # lines = lines[lines.geometry.length > 0.5].copy()

    # Build endpoints and cluster
    coords_xy = _segment_endpoints(lines)
    if coords_xy.size == 0:
        return ConnectivityResult(one_value=float("nan"), one_value_name=one_value, metrics={})

    labels = _cluster_endpoints(coords_xy, snap_tolerance=snap_tolerance_m)
    G = _build_graph(lines, labels, coords_xy)

    metrics = _compute_metrics(G)
    value = metrics.get(one_value)
    # Some metrics may be None; fall back to giant component ratio
    if value is None:
        value = metrics.get("giant_component_ratio")

    return ConnectivityResult(one_value=float(value), one_value_name=one_value, metrics=metrics)

In [ ]:
path = "gis_files/road_network/pedestrian_accessible_road_network_merged.geojson"
gdf = gpd.read_file(path)
parks_boundaries = gpd.read_file('gis_files/parks_buffer_2km.geojson')

In [ ]:
# plot gdf and parks_boundaries
ax = gdf.plot(figsize=(10, 10), color='lightgrey', linewidth=0.5)
parks_boundaries.boundary.plot(ax=ax, color='red', linewidth=1)
ax.set_title('Pedestrian Accessible Road Network and Park Boundaries')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

In [ ]:
# intersect gdf with parks_boundaries with the attributes of parks_boundaries retained
gdf = gpd.overlay(gdf, parks_boundaries, how='intersection')
gdf['layer'].value_counts()

In [ ]:
# plot gdf and parks_boundaries, gdf colored by 'layer'
ax = gdf.plot(figsize=(10, 10), column='layer', cmap='Set2', legend=True, linewidth=0.25)
parks_boundaries.boundary.plot(ax=ax, color='red', linewidth=1)
ax.set_title('Pedestrian Accessible Road Network and Park Boundaries')
# hide the axis labels for cleaner look
ax.set_axis_off()

In [ ]:
for park in gdf['layer'].unique():
    sub_gdf = gdf[gdf['layer'] == park]
    result = connectivity_from_geojson(sub_gdf, snap_tolerance_m=5.0, one_value="giant_component_ratio")
    print(f"Park: {park}")
    print(f"Connectivity ({result.one_value_name}): {result.one_value}")
    print("Metrics:")
    for k, v in result.metrics.items():
        print(f"  {k}: {v}")
    print("\n")